In [1]:
import os
import glob
import cv2
import csv
import mediapipe as mp

In [ ]:
# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_draw = mp.solutions.drawing_utils

# Paths
input_root_dir = "../../Video_Data"
output_root_dir = "Extracted_Features"

# Ensure the output root directory exists
os.makedirs(output_root_dir, exist_ok=True)

# Iterate through each person's folder in the input directory
for person in os.listdir(input_root_dir):
    person_input_path = os.path.join(input_root_dir, person)
    if not os.path.isdir(person_input_path):
        continue  # Skip if it's not a directory
    print(f"Processing data for person: {person}")   

    # Look for subfolders under the person's directory
    subfolders = [subfolder for subfolder in os.listdir(person_input_path) if os.path.isdir(os.path.join(person_input_path, subfolder))]

    for subfolder in subfolders:
        subfolder_path = os.path.join(person_input_path, subfolder)

        # Look for the 'Pose' video inside the subfolder
        video_paths = glob.glob(os.path.join(subfolder_path, "*Pose*.mp4"))
        if not video_paths:
            print(f"No 'Pose' videos found in subfolder {subfolder} for {person}")
            continue

        # Create a corresponding output folder for the subfolder
        subfolder_output_path = os.path.join(output_root_dir, person, subfolder)
        os.makedirs(subfolder_output_path, exist_ok=True)

        # Corrected output CSV filename format: "User_PL_Session.csv"
        data_file = os.path.join(subfolder_output_path, f"{person}_PL_{subfolder}.csv")

        # Process each video in the subfolder
        for video_path in video_paths:
            cap = cv2.VideoCapture(video_path)
            frame_count = 0

            with open(data_file, mode='w', newline='') as file:
                csv_writer = csv.writer(file)

                # Write header: Frame ID and landmark coordinates
                header = ["frame"] + [f"{i}_x" for i in range(33)] + [f"{i}_y" for i in range(33)] + [f"{i}_z" for i in range(33)]
                csv_writer.writerow(header)

                while True:
                    success, img = cap.read()
                    if not success:
                        break

                    # Process the frame with MediaPipe Pose
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    results = pose.process(img_rgb)

                    if results.pose_landmarks:
                        # Save the pose landmarks for this frame
                        row = [frame_count]
                        for lm in results.pose_landmarks.landmark:
                            row.extend([lm.x, lm.y, lm.z])  # Add x, y, z coordinates

                        csv_writer.writerow(row)

                    frame_count += 1

            cap.release()
            print(f"✅ Landmarks saved for {person} ({subfolder}) at {data_file}")

print("🎯 Processing completed.")
